In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', None)       # mostra todas as colunas
pd.set_option('display.max_rows', 100)           # mostra mais linhas
pd.set_option('display.max_colwidth', None)      # não corta texto
pd.set_option('display.width', 2000)             # evita quebra de linha

In [3]:
emp = pd.read_csv('../data/raw/empresas_sp.csv', sep=';')
emp['cod_classe'] = emp['cod_classe'].astype('Int64')

In [4]:
print("=== SHAPE:", emp.shape)
print("=== COLUNAS:", emp.columns.tolist())
print("=== TIPOS:\n", emp.dtypes)
print("=== NULOS:\n", emp.isnull().sum())
print(emp.head(3))

=== SHAPE: (300311, 5)
=== COLUNAS: ['cod_munibge', 'cod_classe', 'cod_porte_empresa', 'opcao_mei', 'empresas_ativas']
=== TIPOS:
 cod_munibge          int64
cod_classe           Int64
cod_porte_empresa    int64
opcao_mei            int64
empresas_ativas      int64
dtype: object
=== NULOS:
 cod_munibge             0
cod_classe           1028
cod_porte_empresa       0
opcao_mei               0
empresas_ativas         0
dtype: int64
   cod_munibge  cod_classe  cod_porte_empresa  opcao_mei  empresas_ativas
0      3510302       86402                  1          2                1
1      3546009       86402                  1          2                1
2      3540606       86402                  1          2                1


In [5]:
emp_dic = pd.read_csv('../data/raw/dicionario_empresas.csv', sep=';', encoding='latin-1')

emp_dic.head(8)

,Id_Variável,Tipo,Descrição,Fonte,Série disponível
0,cod_porte,Número,"Corresponde às categorias referentes ao porte; onde:\n""1 -Micro empresa ""\n""3 - Empresa de pequeno porte ""\n""5 - Demais ""\n","Cadastro Nacional da Pessoa Jurídica - CNPJ, Secretaria Especial da Receita Federal do Brasil",Corresponde à data de abertura das empresas até a data atual do painel
1,empresas_ativas,Número,Total de empresas ativas,"Cadastro Nacional da Pessoa Jurídica - CNPJ, Secretaria Especial da Receita Federal do Brasil; Fundação Seade",Corresponde à data de abertura das empresas até a data atual do painel
2,cod_classe_ibge,Texto,"Corresponde ao código de classe da ""Tabela de Classificação Nacional de Atividades Econômicas"" - CNAE- IBGE",IBGE; Fundação Seade,Corresponde à data de abertura das empresas até a data atual do painel
3,cod_mun_ibge,Texto,"Corresponde ao código do município da ""Tabela de Códigos de Municípios""-IBGE",IBGE; Fundação Seade,Corresponde à data de abertura das empresas até a data atual do painel
4,opcao_mei,Texto,"Corresponde às categorias do regime fiscal; onde:\n""1 -Sim""\n""2 - Não""\n""0"" - Não se aplica""","Cadastro Nacional da Pessoa Jurídica - CNPJ, Secretaria Especial da Receita Federal do Brasil; Fundação Seade",Corresponde à data de abertura das empresas até a data atual do painel
5,cod_natjur_receitafederal,Texto,A natureza jurídica identifica a forma jurídico-institucional das entidades e é usada para definir sua constituição legal nos cadastros da administração pública do país.,NaN,NaN
6,Tempo_ Medio_ de_ Registro_horas,Número,"É o período necessário para a formalização legal da empresa junto aos órgãos competentes, incluindo a emissão do CNPJ pela Receita Federal.",Bases de solicitações da Rede Nacional para a Simplificação do Registro e da Legalização de Empresas e Negócios (REDESIM); Fundação Seade,2019 a 2024
7,Tempo_ Medio_ de_ Abertura_horas,Número,"refere-se ao período necessário para que uma empresa obtenha todas as autorizações exigidas para iniciar suas atividades de forma legal, assegurando o funcionamento conforme as normas e as atividades previstas.",Bases de solicitações da Rede Nacional para a Simplificação do Registro e da Legalização de Empresas e Negócios (REDESIM); Fundação Seade,2019 a 2024


In [6]:
print(emp['cod_porte_empresa'].value_counts())
print(emp['opcao_mei'].value_counts())

cod_porte_empresa
1    211503
5     53962
3     34846
Name: count, dtype: int64
opcao_mei
2    142245
0    115935
1     42131
Name: count, dtype: int64


In [7]:
emp_total = emp.groupby('cod_munibge')['empresas_ativas'].sum().reset_index()
emp_total = emp_total.rename(columns={'empresas_ativas': 'total_empresas'})

emp_mei = emp[emp['opcao_mei'] == 1] \
    .groupby('cod_munibge')['empresas_ativas'] \
    .sum() \
    .reset_index()

emp_mei = emp_mei.rename(columns={'empresas_ativas': 'total_mei'})

emp_mun = emp_total.merge(emp_mei, on='cod_munibge', how='left')
emp_mun['total_mei'] = emp_mun['total_mei'].fillna(0)

emp_mun['pct_mei'] = emp_mun['total_mei'] / emp_mun['total_empresas'] * 100

emp_mun['cod_munibge'] = emp_mun['cod_munibge'].astype(str)

In [8]:
print(emp_mun.shape)
print(emp_mun.head())

(645, 4)
  cod_munibge  total_empresas  total_mei    pct_mei
0     3500105            4857        128   2.635372
1     3500204             463         49  10.583153
2     3500303            4207        563  13.382458
3     3500402             897        124  13.823857
4     3500501            3632        395  10.875551


In [9]:
emp_mun['pct_mei'].describe()

count    645.000000
mean      12.520916
std        2.935040
min        2.635372
25%       10.576923
50%       12.261687
75%       14.229182
max       22.485207
Name: pct_mei, dtype: float64

In [10]:
emp_porte = emp.groupby(['cod_munibge', 'cod_porte_empresa'])['empresas_ativas'] \
    .sum() \
    .reset_index()

In [11]:
emp_porte_pivot = emp_porte.pivot(
    index='cod_munibge',
    columns='cod_porte_empresa',
    values='empresas_ativas'
).reset_index()

In [12]:
emp_porte_pivot = emp_porte_pivot.rename(columns={
    1: 'total_micro',
    3: 'total_pequena',
    5: 'total_demais'
})

In [13]:
emp_porte_pivot = emp_porte_pivot.fillna(0)

In [14]:
emp_porte_pivot[['total_micro', 'total_pequena', 'total_demais']] = \
    emp_porte_pivot[['total_micro', 'total_pequena', 'total_demais']].astype(int)

In [15]:
emp_porte_pivot['total_empresas_check'] = (
    emp_porte_pivot['total_micro'] +
    emp_porte_pivot['total_pequena'] +
    emp_porte_pivot['total_demais']
)

In [16]:
emp_porte_pivot[['total_empresas_check']].describe()

cod_porte_empresa,total_empresas_check
count,6.450000e+02
mean,1.190476e+04
std,1.006786e+05
min,1.200000e+02
25%,6.740000e+02
50%,1.752000e+03
75%,5.465000e+03
max,2.511125e+06


In [17]:
emp_porte_pivot['pct_micro'] = emp_porte_pivot['total_micro'] / emp_porte_pivot['total_empresas_check'] * 100
emp_porte_pivot['pct_pequena'] = emp_porte_pivot['total_pequena'] / emp_porte_pivot['total_empresas_check'] * 100
emp_porte_pivot['pct_demais'] = emp_porte_pivot['total_demais'] / emp_porte_pivot['total_empresas_check'] * 100

In [18]:
emp_porte_pivot.columns.name = None
emp_porte_pivot.head(10)

,cod_munibge,total_micro,total_pequena,total_demais,total_empresas_check,pct_micro,pct_pequena,pct_demais
0,3500105,4590,38,229,4857,94.502779,0.782376,4.714845
1,3500204,449,4,10,463,96.976242,0.863931,2.159827
2,3500303,3941,118,148,4207,93.677205,2.804849,3.517946
3,3500402,842,7,48,897,93.868450,0.780379,5.351171
4,3500501,3489,32,111,3632,96.062775,0.881057,3.056167
5,3500550,1037,11,33,1081,95.929695,1.017576,3.052729
6,3500600,967,6,47,1020,94.803922,0.588235,4.607843
7,3500709,6458,67,165,6690,96.532138,1.001495,2.466368
8,3500758,593,11,22,626,94.728435,1.757188,3.514377
9,3500808,420,4,15,439,95.671982,0.911162,3.416856


In [19]:
emp_porte_pivot[['pct_micro', 'pct_pequena', 'pct_demais']].describe()

,pct_micro,pct_pequena,pct_demais
count,645.000000,645.000000,645.000000
mean,95.033729,1.335983,3.630288
std,2.057158,0.818479,1.569720
min,78.267469,0.000000,0.462963
25%,94.169096,0.780379,2.586207
50%,95.228216,1.176471,3.576590
75%,96.303142,1.729780,4.361874
max,99.382716,5.641127,16.091405


In [20]:
pd.crosstab(emp['cod_porte_empresa'], emp['opcao_mei'], values=emp['empresas_ativas'], aggfunc='sum')

opcao_mei,0,1,2
cod_porte_empresa,,,
1,811521.0,960254.0,5230106.0
3,70467.0,NaN,147794.0
5,393062.0,NaN,65364.0


In [21]:
total_micro = emp[emp['cod_porte_empresa'] == 1]['empresas_ativas'].sum()
total_mei = emp[emp['opcao_mei'] == 1]['empresas_ativas'].sum()

print(total_mei / total_micro)

0.13714229076443887


In [22]:
total_empresas_sp = emp['empresas_ativas'].sum()
print(total_empresas_sp)

7678568


In [23]:
total_micro_sp = emp[emp['cod_porte_empresa'] == 1]['empresas_ativas'].sum()
total_pequena_sp = emp[emp['cod_porte_empresa'] == 3]['empresas_ativas'].sum()
total_demais_sp = emp[emp['cod_porte_empresa'] == 5]['empresas_ativas'].sum()


print("Total empresas:", total_empresas_sp)
print("Micro:", total_micro_sp)
print("Pequena:", total_pequena_sp)
print("Demais:", total_demais_sp)

Total empresas: 7678568
Micro: 7001881
Pequena: 218261
Demais: 458426


In [24]:
print("Micro %:", total_micro_sp / total_empresas_sp * 100)
print("Pequena %:", total_pequena_sp / total_empresas_sp * 100)
print("Demais %:", total_demais_sp / total_empresas_sp * 100)

Micro %: 91.18732815806281
Pequena %: 2.842470106405257
Demais %: 5.970201735531937


In [25]:
total_mei_sp = emp[emp['opcao_mei'] == 1]['empresas_ativas'].sum()
print(total_mei_sp)

960254


In [26]:
print("MEI %:", total_mei_sp / total_empresas_sp * 100)

MEI %: 12.505639072285355


In [27]:
emp.groupby('cod_porte_empresa')['empresas_ativas'].sum()

cod_porte_empresa
1    7001881
3     218261
5     458426
Name: empresas_ativas, dtype: int64

In [28]:
resumo = pd.DataFrame({
    'categoria': ['Micro', 'Pequena', 'Demais', 'MEI'],
    'total': [
        total_micro_sp,
        total_pequena_sp,
        total_demais_sp,
        total_mei_sp
    ]
})

resumo['percentual'] = resumo['total'] / total_empresas_sp * 100

resumo

,categoria,total,percentual
0,Micro,7001881,91.187328
1,Pequena,218261,2.842470
2,Demais,458426,5.970202
3,MEI,960254,12.505639


# IPDM

In [29]:
ipdm = pd.read_csv('../data/raw/ipdm.csv', sep=';', encoding='latin-1')
ipdm['Valor'] = ipdm['Valor'].str.replace(',', '.').astype(float)
ipdm['Valor Estado'] = ipdm['Valor Estado'].str.replace(',', '.').astype(float)

print(ipdm.shape)
print(ipdm.columns.tolist())
print(ipdm.head())

(61920, 11)
['cod_ibge', 'Municipio', 'Valor', 'Ano', 'Tipo', 'Valor Estado', 'Indicador 1', 'Indicador 2', 'Indicador 3', 'Indicador 4', 'Indicador 5']
   cod_ibge   Municipio  Valor   Ano     Tipo  Valor Estado Indicador 1 Indicador 2 Indicador 3 Indicador 4        Indicador 5
0   3500105  Adamantina  0.365  2014  Riqueza         0.457         NaN         NaN         NaN         NaN  Indicador Riqueza
1   3500105  Adamantina  0.365  2016  Riqueza         0.438         NaN         NaN         NaN         NaN  Indicador Riqueza
2   3500105  Adamantina  0.382  2018  Riqueza         0.451         NaN         NaN         NaN         NaN  Indicador Riqueza
3   3500105  Adamantina  0.378  2020  Riqueza         0.439         NaN         NaN         NaN         NaN  Indicador Riqueza
4   3500105  Adamantina  0.359  2022  Riqueza         0.441         NaN         NaN         NaN         NaN  Indicador Riqueza


In [30]:
ipdm['Tipo'].unique()

array(['Riqueza', 'Longevidade', 'Escolaridade', 'IPDM'], dtype=object)

In [31]:
ipdm['Ano'].max()

2024

In [32]:
ipdm_2024 = ipdm[ipdm['Ano'] == 2024].copy()

In [33]:
ipdm_final = ipdm_2024[
    ipdm_2024['Indicador 5'].notna()
].copy()

In [34]:
ipdm_final

,cod_ibge,Municipio,Valor,Ano,Tipo,Valor Estado,Indicador 1,Indicador 2,Indicador 3,Indicador 4,Indicador 5
5,3500105,Adamantina,0.381,2024,Riqueza,0.464,NaN,NaN,NaN,NaN,Indicador Riqueza
11,3500204,Adolfo,0.427,2024,Riqueza,0.464,NaN,NaN,NaN,NaN,Indicador Riqueza
17,3500303,Aguaí,0.387,2024,Riqueza,0.464,NaN,NaN,NaN,NaN,Indicador Riqueza
23,3500402,Águas da Prata,0.326,2024,Riqueza,0.464,NaN,NaN,NaN,NaN,Indicador Riqueza
29,3500501,Águas de Lindóia,0.421,2024,Riqueza,0.464,NaN,NaN,NaN,NaN,Indicador Riqueza
...,...,...,...,...,...,...,...,...,...,...,...
11585,3557006,Votorantim,0.638,2024,Escolaridade,0.569,NaN,NaN,NaN,NaN,Indicador Escolaridade
11591,3557105,Votuporanga,0.701,2024,Escolaridade,0.569,NaN,NaN,NaN,NaN,Indicador Escolaridade
11597,3557154,Zacarias,0.679,2024,Escolaridade,0.569,NaN,NaN,NaN,NaN,Indicador Escolaridade
11603,3557204,Chavantes,0.523,2024,Escolaridade,0.569,NaN,NaN,NaN,NaN,Indicador Escolaridade


In [35]:
ipdm_pivot = ipdm_final.pivot(
    index='cod_ibge',
    columns='Tipo',
    values='Valor'
).reset_index()

In [36]:
ipdm_total = ipdm_2024[ipdm_2024['Tipo'] == 'IPDM'][['cod_ibge', 'Valor']].copy()
ipdm_total = ipdm_total.rename(columns={'Valor': 'ipdm_total'})

In [37]:
ipdm_final_df = ipdm_pivot.merge(ipdm_total, on='cod_ibge')

In [38]:
ipdm_final_df

,cod_ibge,Escolaridade,Longevidade,Riqueza,ipdm_total
0,3500105,0.658,0.784,0.381,0.608
1,3500204,0.746,0.778,0.427,0.650
2,3500303,0.480,0.733,0.387,0.533
3,3500402,0.527,0.778,0.326,0.544
4,3500501,0.695,0.820,0.421,0.645
...,...,...,...,...,...
640,3557006,0.638,0.685,0.366,0.563
641,3557105,0.701,0.774,0.444,0.640
642,3557154,0.679,0.656,0.322,0.552
643,3557204,0.523,0.616,0.376,0.505


In [39]:
print("=== SHAPE:", ipdm_final_df.shape)
print("=== COLUNAS:", ipdm_final_df.columns.tolist())
print("=== TIPOS:\n", ipdm_final_df.dtypes)
print("=== NULOS:\n", ipdm_final_df.isnull().sum())
print(ipdm_final_df.head(3))

=== SHAPE: (645, 5)
=== COLUNAS: ['cod_ibge', 'Escolaridade', 'Longevidade', 'Riqueza', 'ipdm_total']
=== TIPOS:
 cod_ibge          int64
Escolaridade    float64
Longevidade     float64
Riqueza         float64
ipdm_total      float64
dtype: object
=== NULOS:
 cod_ibge        0
Escolaridade    0
Longevidade     0
Riqueza         0
ipdm_total      0
dtype: int64
   cod_ibge  Escolaridade  Longevidade  Riqueza  ipdm_total
0   3500105         0.658        0.784    0.381       0.608
1   3500204         0.746        0.778    0.427       0.650
2   3500303         0.480        0.733    0.387       0.533


In [40]:
# garantir que as chaves estejam no mesmo formato
emp_porte_pivot['cod_munibge'] = emp_porte_pivot['cod_munibge'].astype(str)
emp_mun['cod_munibge'] = emp_mun['cod_munibge'].astype(str)
ipdm_final_df['cod_ibge'] = ipdm_final_df['cod_ibge'].astype(str)


In [41]:
df_empresas = emp_porte_pivot.merge(
    emp_mun[['cod_munibge', 'total_mei', 'pct_mei']],
    on='cod_munibge',
    how='left'
)

In [42]:
df_empresas

,cod_munibge,total_micro,total_pequena,total_demais,total_empresas_check,pct_micro,pct_pequena,pct_demais,total_mei,pct_mei
0,3500105,4590,38,229,4857,94.502779,0.782376,4.714845,128,2.635372
1,3500204,449,4,10,463,96.976242,0.863931,2.159827,49,10.583153
2,3500303,3941,118,148,4207,93.677205,2.804849,3.517946,563,13.382458
3,3500402,842,7,48,897,93.868450,0.780379,5.351171,124,13.823857
4,3500501,3489,32,111,3632,96.062775,0.881057,3.056167,395,10.875551
...,...,...,...,...,...,...,...,...,...,...
640,3557006,16973,395,485,17853,95.070856,2.212513,2.716630,2579,14.445751
641,3557105,14052,259,662,14973,93.848928,1.729780,4.421292,1379,9.209911
642,3557154,282,3,30,315,89.523810,0.952381,9.523810,37,11.746032
643,3557204,1885,13,59,1957,96.320899,0.664282,3.014819,282,14.409811


In [43]:
print("=== SHAPE:", df_empresas.shape)
print("=== COLUNAS:", df_empresas.columns.tolist())
print("=== TIPOS:\n", df_empresas.dtypes)
print("=== NULOS:\n", df_empresas.isnull().sum())
print(df_empresas.head(3))

=== SHAPE: (645, 10)
=== COLUNAS: ['cod_munibge', 'total_micro', 'total_pequena', 'total_demais', 'total_empresas_check', 'pct_micro', 'pct_pequena', 'pct_demais', 'total_mei', 'pct_mei']
=== TIPOS:
 cod_munibge              object
total_micro               int32
total_pequena             int32
total_demais              int32
total_empresas_check      int32
pct_micro               float64
pct_pequena             float64
pct_demais              float64
total_mei                 int64
pct_mei                 float64
dtype: object
=== NULOS:
 cod_munibge             0
total_micro             0
total_pequena           0
total_demais            0
total_empresas_check    0
pct_micro               0
pct_pequena             0
pct_demais              0
total_mei               0
pct_mei                 0
dtype: int64
  cod_munibge  total_micro  total_pequena  total_demais  total_empresas_check  pct_micro  pct_pequena  pct_demais  total_mei    pct_mei
0     3500105         4590             38    

In [44]:
municipios = ipdm_2024[['cod_ibge', 'Municipio']].drop_duplicates()
municipios['cod_ibge'] = municipios['cod_ibge'].astype(str)
municipios.groupby('cod_ibge').size().value_counts()

1    645
Name: count, dtype: int64

In [45]:
ipdm_final_df = ipdm_final_df.merge(
    municipios,
    on='cod_ibge',
    how='left'
)

In [46]:
df_final = df_empresas.merge(
    ipdm_final_df,
    left_on='cod_munibge',
    right_on='cod_ibge',
    how='inner'
)

In [47]:
df_final = df_final.drop(columns=['cod_ibge'])

In [48]:
print("=== SHAPE:", df_final.shape)
print("=== COLUNAS:", df_final.columns.tolist())
print("=== TIPOS:\n", df_final.dtypes)
print("=== NULOS:\n", df_final.isnull().sum())
print(df_final.head(3))

=== SHAPE: (645, 15)
=== COLUNAS: ['cod_munibge', 'total_micro', 'total_pequena', 'total_demais', 'total_empresas_check', 'pct_micro', 'pct_pequena', 'pct_demais', 'total_mei', 'pct_mei', 'Escolaridade', 'Longevidade', 'Riqueza', 'ipdm_total', 'Municipio']
=== TIPOS:
 cod_munibge              object
total_micro               int32
total_pequena             int32
total_demais              int32
total_empresas_check      int32
pct_micro               float64
pct_pequena             float64
pct_demais              float64
total_mei                 int64
pct_mei                 float64
Escolaridade            float64
Longevidade             float64
Riqueza                 float64
ipdm_total              float64
Municipio                object
dtype: object
=== NULOS:
 cod_munibge             0
total_micro             0
total_pequena           0
total_demais            0
total_empresas_check    0
pct_micro               0
pct_pequena             0
pct_demais              0
total_mei        

In [49]:
(df_final['total_micro'] + df_final['total_pequena'] + df_final['total_demais']) - df_final['total_empresas_check']

0      0
1      0
2      0
3      0
4      0
      ..
640    0
641    0
642    0
643    0
644    0
Length: 645, dtype: int32

In [50]:
(df_final['pct_micro'] + df_final['pct_pequena'] + df_final['pct_demais'])

0      100.0
1      100.0
2      100.0
3      100.0
4      100.0
       ...  
640    100.0
641    100.0
642    100.0
643    100.0
644    100.0
Length: 645, dtype: float64

In [51]:
df_final[['ipdm_total', 'Riqueza', 'Escolaridade', 'Longevidade']].describe()

,ipdm_total,Riqueza,Escolaridade,Longevidade
count,645.000000,645.000000,645.000000,645.000000
mean,0.565288,0.387343,0.595840,0.712722
std,0.049477,0.066329,0.086125,0.072075
min,0.413000,0.201000,0.352000,0.345000
25%,0.531000,0.344000,0.537000,0.668000
50%,0.562000,0.384000,0.591000,0.718000
75%,0.600000,0.431000,0.650000,0.756000
max,0.729000,0.621000,0.960000,0.916000


In [52]:
import plotly.express as px

fig = px.scatter(
    df_final,
    x='pct_demais',
    y='ipdm_total',
    hover_name='Municipio',
    title='% de pct_demais × Desenvolvimento (IPDM)',
    labels={
        'pct_demais': '% de mei',
        'ipdm_total': 'IPDM'
    }
)

#fig.show(renderer="browser")

In [53]:
df_final[['pct_micro', 'pct_pequena' , 'pct_mei', 'pct_demais', 'ipdm_total']].corr()

,pct_micro,pct_pequena,pct_mei,pct_demais,ipdm_total
pct_micro,1.000000,-0.723920,0.191343,-0.933061,-0.165293
pct_pequena,-0.723920,1.000000,-0.121572,0.427298,0.228973
pct_mei,0.191343,-0.121572,1.000000,-0.187370,-0.375085
pct_demais,-0.933061,0.427298,-0.187370,1.000000,0.097231
ipdm_total,-0.165293,0.228973,-0.375085,0.097231,1.000000


In [54]:
df_final[['pct_micro', 'pct_pequena' , 'pct_mei', 'pct_demais', 'total_empresas_check','ipdm_total', 'Riqueza', 'Escolaridade', 'Longevidade']].corr()

,pct_micro,pct_pequena,pct_mei,pct_demais,total_empresas_check,ipdm_total,Riqueza,Escolaridade,Longevidade
pct_micro,1.000000,-0.723920,0.191343,-0.933061,-0.221434,-0.165293,-0.329730,-0.008761,-0.026964
pct_pequena,-0.723920,1.000000,-0.121572,0.427298,0.217979,0.228973,0.363519,0.057734,0.068403
pct_mei,0.191343,-0.121572,1.000000,-0.187370,-0.000616,-0.375085,-0.169490,-0.355066,-0.192353
pct_demais,-0.933061,0.427298,-0.187370,1.000000,0.176537,0.097231,0.242575,-0.018622,-0.000329
total_empresas_check,-0.221434,0.217979,-0.000616,0.176537,1.000000,0.027701,0.103540,-0.040014,0.010185
ipdm_total,-0.165293,0.228973,-0.375085,0.097231,0.027701,1.000000,0.543255,0.762101,0.648461
Riqueza,-0.329730,0.363519,-0.169490,0.242575,0.103540,0.543255,1.000000,0.133926,0.038457
Escolaridade,-0.008761,0.057734,-0.355066,-0.018622,-0.040014,0.762101,0.133926,1.000000,0.250964
Longevidade,-0.026964,0.068403,-0.192353,-0.000329,0.010185,0.648461,0.038457,0.250964,1.000000


In [55]:
df_final[['pct_mei', 'Escolaridade']].corr()

,pct_mei,Escolaridade
pct_mei,1.000000,-0.355066
Escolaridade,-0.355066,1.000000


In [56]:
df_final

,cod_munibge,total_micro,total_pequena,total_demais,total_empresas_check,pct_micro,pct_pequena,pct_demais,total_mei,pct_mei,Escolaridade,Longevidade,Riqueza,ipdm_total,Municipio
0,3500105,4590,38,229,4857,94.502779,0.782376,4.714845,128,2.635372,0.658,0.784,0.381,0.608,Adamantina
1,3500204,449,4,10,463,96.976242,0.863931,2.159827,49,10.583153,0.746,0.778,0.427,0.650,Adolfo
2,3500303,3941,118,148,4207,93.677205,2.804849,3.517946,563,13.382458,0.480,0.733,0.387,0.533,Aguaí
3,3500402,842,7,48,897,93.868450,0.780379,5.351171,124,13.823857,0.527,0.778,0.326,0.544,Águas da Prata
4,3500501,3489,32,111,3632,96.062775,0.881057,3.056167,395,10.875551,0.695,0.820,0.421,0.645,Águas de Lindóia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,3557006,16973,395,485,17853,95.070856,2.212513,2.716630,2579,14.445751,0.638,0.685,0.366,0.563,Votorantim
641,3557105,14052,259,662,14973,93.848928,1.729780,4.421292,1379,9.209911,0.701,0.774,0.444,0.640,Votuporanga
642,3557154,282,3,30,315,89.523810,0.952381,9.523810,37,11.746032,0.679,0.656,0.322,0.552,Zacarias
643,3557204,1885,13,59,1957,96.320899,0.664282,3.014819,282,14.409811,0.523,0.616,0.376,0.505,Chavantes


In [57]:
import duckdb

con = duckdb.connect("../data/processed/empresas_sp.duckdb")

In [58]:
con.execute("""
CREATE OR REPLACE TABLE empresas_municipio AS
SELECT
    cod_munibge,
    total_micro,
    total_pequena,
    total_demais,
    total_empresas_check AS total_empresas,
    total_mei,
    pct_micro,
    pct_pequena,
    pct_demais,
    pct_mei
FROM df_empresas
""")

In [59]:
con.execute("""
CREATE OR REPLACE TABLE ipdm_municipio AS
SELECT
    cod_ibge AS cod_munibge,
    Municipio AS municipio,
    Escolaridade AS escolaridade,
    Longevidade AS longevidade,
    Riqueza AS riqueza,
    ipdm_total
FROM ipdm_final_df
""")

In [60]:
con.execute("""
CREATE OR REPLACE VIEW vw_municipios_analise AS
SELECT
    e.cod_munibge,
    i.municipio,

    -- Empresas
    e.total_micro,
    e.total_pequena,
    e.total_demais,
    e.total_mei,
    e.total_empresas,
    e.pct_micro,
    e.pct_pequena,
    e.pct_demais,
    e.pct_mei,

    -- IPDM
    i.escolaridade,
    i.longevidade,
    i.riqueza,
    i.ipdm_total

FROM empresas_municipio e
LEFT JOIN ipdm_municipio i
    ON e.cod_munibge = i.cod_munibge
""")

In [61]:
df_sql = con.execute("""
SELECT *
FROM vw_municipios_analise
""").fetchdf()

In [62]:
df_sql

,cod_munibge,municipio,total_micro,total_pequena,total_demais,total_mei,total_empresas,pct_micro,pct_pequena,pct_demais,pct_mei,escolaridade,longevidade,riqueza,ipdm_total
0,3500105,Adamantina,4590,38,229,128,4857,94.502779,0.782376,4.714845,2.635372,0.658,0.784,0.381,0.608
1,3500303,Aguaí,3941,118,148,563,4207,93.677205,2.804849,3.517946,13.382458,0.480,0.733,0.387,0.533
2,3500402,Águas da Prata,842,7,48,124,897,93.868450,0.780379,5.351171,13.823857,0.527,0.778,0.326,0.544
3,3500501,Águas de Lindóia,3489,32,111,395,3632,96.062775,0.881057,3.056167,10.875551,0.695,0.820,0.421,0.645
4,3500550,Águas de Santa Bárbara,1037,11,33,135,1081,95.929695,1.017576,3.052729,12.488437,0.683,0.731,0.455,0.623
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,3539905,Poloni,736,3,21,92,760,96.842105,0.394737,2.763158,12.105263,0.615,0.807,0.348,0.590
641,3541208,Presidente Bernardes,1489,15,62,188,1566,95.083014,0.957854,3.959132,12.005109,0.556,0.668,0.320,0.515
642,3545704,Santa Albertina,565,14,13,86,592,95.439189,2.364865,2.195946,14.527027,0.627,0.549,0.399,0.525
643,3556800,Viradouro,2348,25,101,252,2474,94.907033,1.010509,4.082458,10.185934,0.484,0.788,0.364,0.545


In [63]:
con.close()